In [42]:
from youtube_transcript_api import YouTubeTranscriptApi
from youtube_transcript_api._errors import TranscriptsDisabled, NoTranscriptFound
from youtube_transcript_api._errors import TranscriptsDisabled
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import ChatOllama,OllamaEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate
from langchain_core.runnables import RunnableParallel,RunnablePassthrough,RunnableLambda
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_openai import OpenAIEmbeddings
from dotenv import load_dotenv
from langchain_nvidia_ai_endpoints import NVIDIAEmbeddings
import os

In [43]:
# embeddingmodel = OllamaEmbeddings(
#     model = "embeddinggemma:latest"
# )



load_dotenv()

# embeddingmodel = OpenAIEmbeddings(
#     api_key=os.getenv("NVIDIA_API_KEY"),
#     base_url="https://integrate.api.nvidia.com/v1",
#     model="nvidia/llama-nemotron-embed-1b-v2"
# )

embeddingmodel = NVIDIAEmbeddings(
    model="nvidia/llama-nemotron-embed-1b-v2",
    api_key=os.getenv("NVIDIA_API_KEY"),
)


model = ChatOpenAI(
    api_key=os.getenv("NVIDIA_API_KEY"),
    base_url="https://integrate.api.nvidia.com/v1",
    model="nvidia/nemotron-3-ultra-550b-a55b"
)

# Indexing (Document Ingestion)

In [13]:


video_id = "D1PcZaeQ2eg"

try:
    youtube_api = YouTubeTranscriptApi()
    # transcript_list = youtube_api.fetch(video_id,languages=["en"])

    # Transcript object
    # original_trans_list = next(iter(transcript_list))

    # print(f"\nLanguage: {transcript_list.language}")

    # original_transcript = " ".join(chunk.text for chunk in transcript_list)

    # print("\n👉 Original Transcript:\n")
    # print(original_transcript)

    # Get all available transcripts
    transcript_data = youtube_api.list(video_id)

    # Get the first available transcript (regardless of language)
    transcript = next(iter(transcript_data))

    # Fetch transcript
    transcript_list = transcript.fetch()

    # Convert to text
    original_transcript = " ".join(chunk.text for chunk in transcript_list)

    # English translation

    # english_trans_list = youtube_api.fetch(video_id,languages=["en"])
    # english_transcript = " ".join(chunk.text for chunk in english_trans_list)

    # print(f"\nLanguage: {english_trans_list.language}")
    # print("\n👉 English Translation:\n")
    # print(english_transcript)

    # # Hindi translation

    # hindi_trans_list = youtube_api.fetch(video_id,languages=["hi"])
    # hindi_transcript = " ".join(chunk.text for chunk in hindi_trans_list)

    # print(f"\nLanguage: {hindi_trans_list.language}")
    # print("\n👉 Hindi Translation:\n")
    # print(hindi_transcript)

except TranscriptsDisabled:
    print("❌ Captions are disabled for this video.")




In [14]:
# from youtube_transcript_api import YouTubeTranscriptApi

# video_id = "D1PcZaeQ2eg"

# youtube_api = YouTubeTranscriptApi()

# # Get all available transcripts
# transcript_list = youtube_api.list(video_id)

# # Get the first available transcript (regardless of language)
# transcript = next(iter(transcript_list))

# # Fetch transcript
# transcript_data = transcript.fetch()

# # Convert to text
# original_transcript = " ".join(chunk.text for chunk in transcript_data)


In [16]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='हाय गाइज़, माय नेम इज़ नितेश एंड यू आर', start=0.0, duration=4.16), FetchedTranscriptSnippet(text='वेलकम टू माय YouTube चैनल। इस वीडियो में', start=1.92, duration=4.56), FetchedTranscriptSnippet(text='भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग', start=4.16, duration=4.56), FetchedTranscriptSnippet(text='ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो', start=6.48, duration=4.72), FetchedTranscriptSnippet(text='पिछले कुछ वीडियोस में जैसा हमने प्लान', start=8.72, duration=5.6), FetchedTranscriptSnippet(text='किया था हम एक चैटबॉट डेवलप कर रहे हैं।', start=11.2, duration=6.24), FetchedTranscriptSnippet(text='और धीरे-धीरे उस चैटबॉट में हम फीचर्स और', start=14.32, duration=5.6), FetchedTranscriptSnippet(text='ऐड करते जा रहे हैं। सबसे पहले हमने एक', start=17.44, duration=4.24), FetchedTranscriptSnippet(text='बेसिक चैटबॉट बनाया था। जहां पे आप एक', start=19.92, duration=4.64), FetchedTranscriptSnippet(text='एलएलएम से बात कर पा रहे थे। फिर उसी में', start=

In [17]:
original_transcript

'हाय गाइज़, माय नेम इज़ नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो पिछले कुछ वीडियोस में जैसा हमने प्लान किया था हम एक चैटबॉट डेवलप कर रहे हैं। और धीरे-धीरे उस चैटबॉट में हम फीचर्स और ऐड करते जा रहे हैं। सबसे पहले हमने एक बेसिक चैटबॉट बनाया था। जहां पे आप एक एलएलएम से बात कर पा रहे थे। फिर उसी में हमने शॉर्ट टर्म मेमोरी का फीचर ऐड किया। सो दैट हमारा चैटबॉट हमारे पास इंटरेक्शंस याद रख पाए। उसके बाद हमने उस चैटबॉट को एक यूआई दिया। आज हम अपने चैटबॉट की एक और प्रॉब्लम सॉल्व करने जा रहे हैं। सबसे पहले मैं आपको वो प्रॉब्लम दिखाता हूं और फिर मैं आपको बताता हूं कि क्या सॉल्यूशन है उस प्रॉब्लम को सॉल्व करने का। ठीक है? तो स्क्रीन पे अभी आपको हमारे चैटबॉट का यूआई दिख रहा होगा। सो यहां पे एक बार पहले चेक कर लेते हैं। चैटबॉट सही से रिप्लाई कर रहा है कि नहीं। यू कैन सी चैटबॉट रिप्लाई कर रहा है। अब देखो मैं क्या प्र्प रहा हूं। मैं क्या क्वेश्चन पूछ रहा हूं। मैं अ मेरे चैटबॉट को बोल रहा हूं दैट यू हैव टू राइट अ 500 

In [18]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='हाय गाइज़, माय नेम इज़ नितेश एंड यू आर', start=0.0, duration=4.16), FetchedTranscriptSnippet(text='वेलकम टू माय YouTube चैनल। इस वीडियो में', start=1.92, duration=4.56), FetchedTranscriptSnippet(text='भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग', start=4.16, duration=4.56), FetchedTranscriptSnippet(text='ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो', start=6.48, duration=4.72), FetchedTranscriptSnippet(text='पिछले कुछ वीडियोस में जैसा हमने प्लान', start=8.72, duration=5.6), FetchedTranscriptSnippet(text='किया था हम एक चैटबॉट डेवलप कर रहे हैं।', start=11.2, duration=6.24), FetchedTranscriptSnippet(text='और धीरे-धीरे उस चैटबॉट में हम फीचर्स और', start=14.32, duration=5.6), FetchedTranscriptSnippet(text='ऐड करते जा रहे हैं। सबसे पहले हमने एक', start=17.44, duration=4.24), FetchedTranscriptSnippet(text='बेसिक चैटबॉट बनाया था। जहां पे आप एक', start=19.92, duration=4.64), FetchedTranscriptSnippet(text='एलएलएम से बात कर पा रहे थे। फिर उसी में', start=

In [5]:
import importlib.metadata

print(importlib.metadata.version("youtube-transcript-api"))

1.2.4


In [6]:
!pip install youtube-transcript-api==1.2.4


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip


In [7]:
# from youtube_transcript_api import YouTubeTranscriptApi

# api = YouTubeTranscriptApi()

# for lang in [["hi"], ["en"]]:
#     try:
#         data = api.fetch("beGTWgMbDAM", languages=lang)
#         print(f"SUCCESS: {lang}")
#     except Exception as e:
#         print(f"FAILED: {lang}")
#         print(e)

In [8]:
# for transcript in transcript_list:
#     print("Language:", transcript.language)
#     print("Code:", transcript.language_code)
#     print(dir(transcript))
    

In [20]:
print(type(transcript_data))
print(transcript_data)

<class 'youtube_transcript_api._transcripts.TranscriptList'>
For this video (D1PcZaeQ2eg) transcripts are available in the following languages:

(MANUALLY CREATED)
None

(GENERATED)
 - hi ("Hindi (auto-generated)")

(TRANSLATION LANGUAGES)
None


In [12]:
end_time = (transcript_list.snippets[6].start) + (transcript_list.snippets[6].duration)
start_time = (transcript_list.snippets[6].start)

print(f"start_time: {start_time}")
print(f"end_time: {end_time}")

start_time: 22.962
end_time: 27.213


In [94]:
!pip show youtube-transcript-api

Name: youtube-transcript-api
Version: 1.2.4
Summary: This is a python API which allows you to get the transcripts/subtitles for a given YouTube video. It also works for automatically generated subtitles, supports translating subtitles and it does not require a headless browser, like other selenium based solutions do!
Home-page: https://github.com/jdepoix/youtube-transcript-api
Author: Jonas Depoix
Author-email: jonas.depoix@web.de
License: MIT
Location: /Users/asifkhan/Desktop/AI_ML/GenAI/LangChain/venv/lib/python3.13/site-packages
Requires: defusedxml, requests
Required-by: 


In [157]:
for i in transcript_list:
    print(i)
    print("\n\n")
    print(i.text)
    print("\n\n")

FetchedTranscriptSnippet(text='हाय गाइस, माय नेम इज नितेश एंड यू आर', start=0.0, duration=4.24)



हाय गाइस, माय नेम इज नितेश एंड यू आर



FetchedTranscriptSnippet(text='वेलकम टू माय YouTube चैनल। इस वीडियो में', start=2.0, duration=4.48)



वेलकम टू माय YouTube चैनल। इस वीडियो में



FetchedTranscriptSnippet(text='भी हम लोग अपना एजेंटिक एआई यूजिंग लंग', start=4.24, duration=4.48)



भी हम लोग अपना एजेंटिक एआई यूजिंग लंग



FetchedTranscriptSnippet(text='ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। आज का', start=6.48, duration=4.64)



ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। आज का



FetchedTranscriptSnippet(text='वीडियो इस प्लेलिस्ट का फोर्थ वीडियो है।', start=8.72, duration=4.16)



वीडियो इस प्लेलिस्ट का फोर्थ वीडियो है।



FetchedTranscriptSnippet(text='अगर आपको याद होगा तो पिछले वीडियो में', start=11.12, duration=4.0)



अगर आपको याद होगा तो पिछले वीडियो में



FetchedTranscriptSnippet(text='हमने एक बहुत डिटेल्ड कंपैरिजन किया था', start=12.88, duration=4.88)



हमने एक बहुत डिटेल्ड कंपैरिजन किया था


### test the embedding_model

In [44]:
embedding = embeddingmodel.embed_query("Hello world")

print(type(embedding))
print(len(embedding))

<class 'list'>
2048


In [40]:
pip install -U langchain-nvidia-ai-endpoints


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Indexing (Text Splitting)

In [31]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000,chunk_overlap=200)

original_chunks = splitter.create_documents([original_transcript])

# english_chunks = splitter.create_documents([english_transcript])

# hindi_chunks = splitter.create_documents([hindi_transcript])

In [32]:
original_chunks

[Document(metadata={}, page_content='हाय गाइज़, माय नेम इज़ नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो पिछले कुछ वीडियोस में जैसा हमने प्लान किया था हम एक चैटबॉट डेवलप कर रहे हैं। और धीरे-धीरे उस चैटबॉट में हम फीचर्स और ऐड करते जा रहे हैं। सबसे पहले हमने एक बेसिक चैटबॉट बनाया था। जहां पे आप एक एलएलएम से बात कर पा रहे थे। फिर उसी में हमने शॉर्ट टर्म मेमोरी का फीचर ऐड किया। सो दैट हमारा चैटबॉट हमारे पास इंटरेक्शंस याद रख पाए। उसके बाद हमने उस चैटबॉट को एक यूआई दिया। आज हम अपने चैटबॉट की एक और प्रॉब्लम सॉल्व करने जा रहे हैं। सबसे पहले मैं आपको वो प्रॉब्लम दिखाता हूं और फिर मैं आपको बताता हूं कि क्या सॉल्यूशन है उस प्रॉब्लम को सॉल्व करने का। ठीक है? तो स्क्रीन पे अभी आपको हमारे चैटबॉट का यूआई दिख रहा होगा। सो यहां पे एक बार पहले चेक कर लेते हैं। चैटबॉट सही से रिप्लाई कर रहा है कि नहीं। यू कैन सी चैटबॉट रिप्लाई कर रहा है। अब देखो मैं क्या प्र्प रहा हूं। मैं क्या क्वेश्चन पूछ रहा हूं। मैं अ मेरे चैटबॉट को ब

In [33]:
len(original_chunks)

26

In [38]:
print(type(original_chunks))

print(type(original_chunks[0]))

print(type(original_chunks[0].page_content))

print(original_chunks[0].page_content)

<class 'list'>
<class 'langchain_core.documents.base.Document'>
<class 'str'>
हाय गाइज़, माय नेम इज़ नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो पिछले कुछ वीडियोस में जैसा हमने प्लान किया था हम एक चैटबॉट डेवलप कर रहे हैं। और धीरे-धीरे उस चैटबॉट में हम फीचर्स और ऐड करते जा रहे हैं। सबसे पहले हमने एक बेसिक चैटबॉट बनाया था। जहां पे आप एक एलएलएम से बात कर पा रहे थे। फिर उसी में हमने शॉर्ट टर्म मेमोरी का फीचर ऐड किया। सो दैट हमारा चैटबॉट हमारे पास इंटरेक्शंस याद रख पाए। उसके बाद हमने उस चैटबॉट को एक यूआई दिया। आज हम अपने चैटबॉट की एक और प्रॉब्लम सॉल्व करने जा रहे हैं। सबसे पहले मैं आपको वो प्रॉब्लम दिखाता हूं और फिर मैं आपको बताता हूं कि क्या सॉल्यूशन है उस प्रॉब्लम को सॉल्व करने का। ठीक है? तो स्क्रीन पे अभी आपको हमारे चैटबॉट का यूआई दिख रहा होगा। सो यहां पे एक बार पहले चेक कर लेते हैं। चैटबॉट सही से रिप्लाई कर रहा है कि नहीं। यू कैन सी चैटबॉट रिप्लाई कर रहा है। अब देखो मैं क्या प्र्प रहा हूं। मैं क्या क्व

In [34]:
original_chunks[2]

Document(metadata={}, page_content='10 सेकंड्स डिपेंडिंग ऑन कितना बड़ा आउटपुट है। एंड सेकंड एक साथ यह पूरा रिस्पांस जब आपके स्क्रीन पे आ रहा है तो इट्स नॉट वेरी रीडेबल। राइट? अगर आपने चैट जीपीटी यूज़ किया होगा तो वहां पर आपने देखा होगा कि वहां पे जब बड़ा आउटपुट होता है तो उसका रिस्पांस इस तरीके से नहीं आता। वहां पे क्या होता है कि कैरेक्टर बाय कैरेक्टर आपका पूरा रिस्पांस टाइप होता हुआ दिखाई देता है। इस फीचर को हम स्ट्रीमिंग बोलते हैं और वही हम लोग इस पर्टिकुलर वीडियो में हमारे चैटबॉट में इंप्लीमेंट करने वाले हैं। सो दैट जब भी हम कोई बड़ा या लॉन्ग आउटपुट एक्सपेक्ट कर रहे हो हमारे चैटबॉट से तो हमें अब्रप्टली एक साथ अचानक से पूरा टेक्स्ट देखने के बजाय टोकन बाय टोकन टाइप राइटर फैशन में सारा आउटपुट दिखाई दे। इट इज अ मच बेटर यूजर एक्सपीरियंस। इनफैक्ट मैंने डेमो प्रिपेयर कर रखा है। मैं आपको साइड बाय साइड दिखाता हूं कि जब आप स्ट्रीमिंग इंप्लीमेंट करते हो किसी चैटबॉट में तो सेम आउटपुट आपको कैसा दिखाई देता है। सो अभी आपको जो मेरे स्क्रीन पे दिखाई दे रहा है वो हमारे ही चैटबॉट का एक डिफरेंट वेरिएंट 

In [35]:
original_chunks[10]

Document(metadata={}, page_content='समझ पाते हो कि अच्छा कोड यहां से स्टार्ट हुआ नेक्स्ट लाइन में यह हो रहा है। उसकी नेक्स्ट लाइन में यह हो रहा है। तो इट इज़ अ मच बेटर यूजर इंटरफ़ेस अह या यूजर एक्सपीरियंस व्हेन यू प्रिंट वर्ड स्टेप बाय स्टेप स्पेशली व्हेन यू हैव रिस्पांसेस फॉर कोड्स एंड ऑल। ठीक है? अह एक और बहुत इंपॉर्टेंट बेनिफिट होता है स्ट्रीमिंग का कि अगर आपको रिस्पांस पसंद नहीं आ रहा चैट GPT का या किसी भी एलएलएम बेस्ड एप्लीकेशन का तो आप मिड वे ब्रेक कर सकते हो उसके रिस्पांस को। को आप स्टॉप कर सकते हो। इससे क्या होगा कि सारे के सारे टोकंस जनरेट नहीं होंगे और आप बीच में ही रिस्पांस को रोक दे रहे हो। तो आप एसेंशियली टोकंस बचा रहे हो और टोकंस बचाने का सिंपल मतलब है आप पैसे बचा रहे हो। बिकॉज़ जितने भी एलएलएम प्रोवाइडर्स हैं वो आपको नंबर ऑफ टोकंस यूसेज के बेसिस पे चार्ज करते हैं। तो यह भी एक बहुत इंपॉर्टेंट चीज है। एंड लास्टली सिर्फ एलएलएम का मैसेज दिखाने के लिए स्ट्रीमिंग यूज़ नहीं होता। आपको कई बार अपडेट्स दिखाने के लिए भी स्ट्रीमिंग यूज किया जाता है। जैसे मान लो आप किसी एi एजेंट से बात क

# Indexing (Embedding Generation & Storing Vector)

In [18]:
#vector_store = Chroma.from_documents(chunks,embedingmodel)

In [19]:
# data = vector_store.get()
# print(data.keys())

In [20]:
# print(data["ids"])

In [21]:
# result = vector_store.get(

#     ids=["af4312cf-b2c7-4929-84aa-13c7981fbfba"]

# )

# print(result['documents'])

In [45]:
vector_store = FAISS.from_documents(original_chunks,embeddingmodel)


In [46]:
vector_store.index_to_docstore_id


{0: '59039c5f-40ac-49e0-9b5d-6742bbb768d3',
 1: '0cac41bc-01fa-46e7-b6b2-b0989c2808b6',
 2: '71b6fdad-d365-4e05-b24c-38c320c389e4',
 3: '997fc791-b663-42ae-bf3e-4bb67f097f63',
 4: '4606e2d8-1554-46cd-9ba7-e1b50558180c',
 5: '63a45117-1c3e-47de-b472-1a5937175cb4',
 6: 'f74f985d-56f4-48a1-a1e9-a602a7cdb92e',
 7: '3ad6f7de-1bb7-4d80-a024-85ea81af1ef3',
 8: '3ab63a59-f82b-4eed-a3dd-2aab04201863',
 9: '16489d3f-576c-4854-bd99-d834f6ce5772',
 10: 'cbf9fc2d-a557-4b93-ab23-3519140633fb',
 11: '7de58e57-adea-4c5c-8622-4abc503ee0c5',
 12: 'bd1cea29-54ea-4260-a8cb-8756dfad4ced',
 13: 'cd1c108d-102b-473b-af0a-8aed272e7607',
 14: '1756828c-24d9-4f97-87d7-86cc85bf13ad',
 15: 'fe3f9e48-5f22-472c-adbd-936796e9a219',
 16: '2a00f70b-acdc-474c-8fc9-69572be15605',
 17: '4ad92980-a7e7-4f5a-a40b-317421a9d1c8',
 18: '940e0245-7cfd-4917-80f1-dcbb7c7c5e90',
 19: '98592439-8dd9-474b-9a45-bcaa67f42df4',
 20: '80fe6da0-97a3-43e5-a5a8-d9030c304e74',
 21: '4ab419ed-4900-4911-933d-74e87ceba9cc',
 22: '7bd4a982-3f99-

In [47]:
vector_store.get_by_ids(['940e0245-7cfd-4917-80f1-dcbb7c7c5e90'])

[Document(id='940e0245-7cfd-4917-80f1-dcbb7c7c5e90', metadata={}, page_content='विल डू इज हम इंस्टेड ऑफ सेविंग दिस ऑब्जेक्ट हम ऑन द गो यह कोड लिखेंगे फॉर मैसेज चंक कॉमा मेटा डेटा इन चैटबॉट डॉट स्ट्रीम अब इस लूप के अंदर हम जाएंगे और हम सिंपली चेक करेंगे कि क्या मैसेज चंक के अंदर कॉनेंट है कि नहीं अगर है तो हम क्या करेंगे उस कॉनेंट को प्रिंट कर देंगे ठीक है लाइक दिस और हम एंड में उस बार के बदले स्पेस को यूज़ करेंगे और नीचे वाले प्रिंट स्टेटमेंट को मैं हटा रहा हूं। सेव किया। दिस इज माय कोड। मैंने रन किया। एंड यू कैन सी हमारा जो आउटपुट है वो स्ट्रीम होने लग गया। तो आई होप आपको समझ में आया। द ओनली डिफरेंस वाज़ कि इंस्टेड ऑफ यूजिंग डॉट इनवोक वी आर यूजिंग डॉट स्ट्रीम। और फिर उस जनरेटर ऑब्जेक्ट के ऊपर हम लूप चला रहे हैं और उसके कंटेंट को प्रिंट कर रहे हैं वि द हेल्प ऑफ प्रिंट स्टेटमेंट। दैट्स द ओनली चेंज दैट वी डिड। अब हमें क्या करना है? यही पार्ट यह पूरी चीज हमें स्ट्रीमलेट में इंप्लीमेंट करनी है। यहां पर हमें कोई चेंजेस नहीं करने। यह हमारा बैक एंड है। तो, हम क्या करेंगे? यह जो पूरा पार्ट हमने 

# Retriever

In [48]:
retriever = vector_store.as_retriever(search_type = "similarity",search_kwargs={"k":4})


In [49]:
retriever

VectorStoreRetriever(tags=['FAISS', 'NVIDIAEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12fa97770>, search_kwargs={'k': 4})

In [52]:
query = "What is the importance of streaming"

In [53]:
retriever.invoke(query)

[Document(id='098ca7e4-6c65-489d-ba62-5a8479a60ea4', metadata={}, page_content='तो, हमने हमारे चार्ट बॉट में एक छोटा सा बट बहुत इंपॉर्टेंट फीचर ऐड किया स्ट्रीमिंग का। इस प्रोसेस में हमने यह भी सीखा कि स्ट्रीमिंग होता क्या है? टेक्निकली स्ट्रीमिंग काम कैसे करता है? और स्ट्रीमिंग क्यों जरूरी है? ठीक है? तो, आई होप आपको यह वीडियो पसंद आया। और अगर वीडियो पसंद आया तो आप प्लीज लाइक करना। अगर आपने चैनल को सब्सक्राइब नहीं किया है तो प्लीज सब्सक्राइब। मिलते हैं नेक्स्ट वीडियो में।'),
 Document(id='3ad6f7de-1bb7-4d80-a024-85ea81af1ef3', metadata={}, page_content='नहीं है टेक्निकल प्रोडक्ट्स यूज करने में। उसके शूज में जाकर देखो कि उसको कैसा फील होगा। उसको लगेगा कि ऐप फ्रीज कर गया है या मे बी काम नहीं कर रहा है और वह ऐप को बंद करके चला जा सकता है। तो बेसिकली आपके ऐप के ऊपर ड्रॉप ऑफ होगा जो कि अच्छी बात नहीं है। इससे बचाता है हमें स्ट्रीमिंग। तुरंत के तुरंत रिस्पांस आने लगता है तो फिर लगता है कि हां सब कुछ सही से काम कर रहा है। सेकंड इंपॉर्टेंट चीज है कि स्ट्रीमिंग मिमिक्स ह्यूमन लाइक कॉन्वर्सेशन। 

# Augmentation

In [54]:
# model = ChatOllama(
#     model= "gemma4:latest",
#     temperature=0.5
# )

In [55]:
prompt = PromptTemplate(
    template="""
you are a helpful assistant who know teach anything very well.
Answer Only from the Provided trascript context.
If the context is Insufficient, just say you don't know.
And you get the transcript in any language then also use your translation to translate in English and repone only in English Language

{context}
Question: {question}""",
input_variables= ['context','question']
)

In [56]:
question = "What is the importance of Streamimg"
retriever_docs = retriever.invoke(question)

In [57]:
retriever_docs

[Document(id='bd1cea29-54ea-4260-a8cb-8756dfad4ced', metadata={}, page_content='ऑफ स्ट्रीमिंग। आगे हम जब एआई एजेंट्स बनाएंगे तो हम स्ट्रीमिंग की हेल्प से इस तरह के अपडेट्स यूजर को दिखाएंगे। ठीक है? तो ये कुछ बेनिफिट्स हैं स्ट्रीमिंग यूज़ करने के। नटशेल में बहुत छोटी सी चीज है। बट इट्स इनक्रेडिबली यूज़फुल। एक यूजर का जो यूजर एक्सपीरियंस होता है किसी भी एप्लीकेशन के ऊपर उसको 10x बढ़ा सकती है, इंप्रूव कर सकती है स्ट्रीमिंग। ठीक है? दैट इज व्हाई इसको पढ़ना और अपने एप्स में यूज़ करना इज़ अ वेरी वेरीरी स्मार्ट डिसीजन। ठीक है? सो नाउ दैट वी नो व्हाट इज़ स्ट्रीमिंग और स्ट्रीमिंग की जरूरत क्यों है? अब हम अपने एग्जिस्टिंग चैटबॉट वाले कोड में स्ट्रीमिंग का फीचर ऐड करेंगे। अब ऑनेस्टली स्ट्रीमिंग को इंप्लीमेंट करने के लिए आपको अपने एकिस्टिंग कोड में बहुत ज्यादा चीजें चेंज करने की जरूरत नहीं है। मैं यहां पर लैंग्राफ्ट के ऑफिशियल डॉक्यूमेंटेशन पे हूं। जहां पर इन लोगों ने बताया है कि कैसे आप स्ट्रीमिंग इंप्लीमेंट कर सकते हो। और इस गिवन एग्जांपल में आप आसानी से देख सकते हो कि जो इकलौता चेंज आपको करना है वो ये

In [58]:
context_text = "\n\n".join(doc.page_content for doc in retriever_docs)
context_text

'ऑफ स्ट्रीमिंग। आगे हम जब एआई एजेंट्स बनाएंगे तो हम स्ट्रीमिंग की हेल्प से इस तरह के अपडेट्स यूजर को दिखाएंगे। ठीक है? तो ये कुछ बेनिफिट्स हैं स्ट्रीमिंग यूज़ करने के। नटशेल में बहुत छोटी सी चीज है। बट इट्स इनक्रेडिबली यूज़फुल। एक यूजर का जो यूजर एक्सपीरियंस होता है किसी भी एप्लीकेशन के ऊपर उसको 10x बढ़ा सकती है, इंप्रूव कर सकती है स्ट्रीमिंग। ठीक है? दैट इज व्हाई इसको पढ़ना और अपने एप्स में यूज़ करना इज़ अ वेरी वेरीरी स्मार्ट डिसीजन। ठीक है? सो नाउ दैट वी नो व्हाट इज़ स्ट्रीमिंग और स्ट्रीमिंग की जरूरत क्यों है? अब हम अपने एग्जिस्टिंग चैटबॉट वाले कोड में स्ट्रीमिंग का फीचर ऐड करेंगे। अब ऑनेस्टली स्ट्रीमिंग को इंप्लीमेंट करने के लिए आपको अपने एकिस्टिंग कोड में बहुत ज्यादा चीजें चेंज करने की जरूरत नहीं है। मैं यहां पर लैंग्राफ्ट के ऑफिशियल डॉक्यूमेंटेशन पे हूं। जहां पर इन लोगों ने बताया है कि कैसे आप स्ट्रीमिंग इंप्लीमेंट कर सकते हो। और इस गिवन एग्जांपल में आप आसानी से देख सकते हो कि जो इकलौता चेंज आपको करना है वो ये है कि अभी तक आप ग्राफ को बनाने के बाद उसको एग्जीक्यूट करने के लिए ग्राफ डॉट\n\

In [59]:
final_prompt = prompt.invoke({"context":context_text,"question":question})

In [60]:
final_prompt

StringPromptValue(text="\nyou are a helpful assistant who know teach anything very well.\nAnswer Only from the Provided trascript context.\nIf the context is Insufficient, just say you don't know.\nAnd you get the transcript in any language then also use your translation to translate in English and repone only in English Language\n\nऑफ स्ट्रीमिंग। आगे हम जब एआई एजेंट्स बनाएंगे तो हम स्ट्रीमिंग की हेल्प से इस तरह के अपडेट्स यूजर को दिखाएंगे। ठीक है? तो ये कुछ बेनिफिट्स हैं स्ट्रीमिंग यूज़ करने के। नटशेल में बहुत छोटी सी चीज है। बट इट्स इनक्रेडिबली यूज़फुल। एक यूजर का जो यूजर एक्सपीरियंस होता है किसी भी एप्लीकेशन के ऊपर उसको 10x बढ़ा सकती है, इंप्रूव कर सकती है स्ट्रीमिंग। ठीक है? दैट इज व्हाई इसको पढ़ना और अपने एप्स में यूज़ करना इज़ अ वेरी वेरीरी स्मार्ट डिसीजन। ठीक है? सो नाउ दैट वी नो व्हाट इज़ स्ट्रीमिंग और स्ट्रीमिंग की जरूरत क्यों है? अब हम अपने एग्जिस्टिंग चैटबॉट वाले कोड में स्ट्रीमिंग का फीचर ऐड करेंगे। अब ऑनेस्टली स्ट्रीमिंग को इंप्लीमेंट करने के लिए आपको अपने एकिस्टिंग कोड में बहुत 

# Generation

In [61]:
answer = model.invoke(final_prompt)

print(answer.content)

Based on the provided transcript, the importance of streaming in LLM-based applications includes:

1.  **Drastically Improves User Experience (10x):** Streaming can improve the user experience of any application by 10x. It prevents users from feeling uncertain or thinking the app has frozen ("app freeze kar gaya hai") during long waits, which otherwise leads to user drop-off.

2.  **Mimics Human-like Conversation:** It builds trust, makes the interaction "feel alive," and keeps the user engaged at every moment (e.g., anticipating "what is next?"), similar to the experience of using ChatGPT.

3.  **Provides Real-time Process Updates (Crucial for AI Agents):** Beyond just streaming LLM tokens, it is essential for showing step-by-step progress of an AI Agent's actions. For example, when booking a movie ticket, streaming allows the agent to show updates like: "Opened BookMyShow," "Selected movie," "Selected seat," "Processing payment," instead of showing nothing for a minute and then sudde

In [62]:
print(answer)

content='Based on the provided transcript, the importance of streaming in LLM-based applications includes:\n\n1.  **Drastically Improves User Experience (10x):** Streaming can improve the user experience of any application by 10x. It prevents users from feeling uncertain or thinking the app has frozen ("app freeze kar gaya hai") during long waits, which otherwise leads to user drop-off.\n\n2.  **Mimics Human-like Conversation:** It builds trust, makes the interaction "feel alive," and keeps the user engaged at every moment (e.g., anticipating "what is next?"), similar to the experience of using ChatGPT.\n\n3.  **Provides Real-time Process Updates (Crucial for AI Agents):** Beyond just streaming LLM tokens, it is essential for showing step-by-step progress of an AI Agent\'s actions. For example, when booking a movie ticket, streaming allows the agent to show updates like: "Opened BookMyShow," "Selected movie," "Selected seat," "Processing payment," instead of showing nothing for a minut

# Building The Chain

In [63]:
# We make a function which can merge the all texts by the retriever

def format_docs(retriever_docs):
    context_text = "\n\n".join(doc.page_content for doc in retriever_docs)
    return context_text

In [64]:
parallel_chain  = RunnableParallel({
    'context':retriever | RunnableLambda(format_docs),
    'question':RunnablePassthrough()
})

In [66]:
parallel_chain.invoke('what is the Straming')

{'context': 'सिर्फ एलएलएम का मैसेज दिखाने के लिए स्ट्रीमिंग यूज़ नहीं होता। आपको कई बार अपडेट्स दिखाने के लिए भी स्ट्रीमिंग यूज किया जाता है। जैसे मान लो आप किसी एi एजेंट से बात कर रहे हो तो एआई एजेंट को आपने बोला कि भाई आप मेरे लिए एक मूवी टिकट बुक कर दो। आप खुद सोचो कैसा लगेगा अगर 1 मिनट तक आपको कुछ भी दिखाई नहीं देगा और सडनली से एजेंट आपको बोलेगा कि आपकी टिकट बुक हो गई है तो वह 1 मिनट जो बीच में होगा ना वहां पे आप बहुत अनसर्टेन होगे और आपको डर लग रहा होगा कि क्या चल रहा है। बट स्ट्रीमिंग की हेल्प से आप क्या कर सकते हो कि एi एजेंट उस पूरे प्रोसेस को कैसे सॉल्व कर रहा है वह स्टेप बाय स्टेप बता सकते हो। जैसे अभी एजेंट ने बुक माय शो ओपन किया। अब उसने कोई पर्टिकुलर मूवी सेलेक्ट की। अब उसने सीट सेलेक्ट की। अब उसने पेमेंट मोड सेलेक्ट किया। अब वो पेमेंट कर रहा है। ये सारे अपडेट्स आप दे सकते हो विद द हेल्प ऑफ स्ट्रीमिंग। आगे हम जब एआई एजेंट्स बनाएंगे तो हम स्ट्रीमिंग की हेल्प से इस तरह के अपडेट्स यूजर को दिखाएंगे। ठीक है? तो ये कुछ बेनिफिट्स हैं स्ट्रीमिंग यूज़ करने के। नटशेल में बहुत छोटी सी 

In [67]:
parser = StrOutputParser()

In [68]:
main_chain = parallel_chain | prompt | model | parser

In [69]:
main_chain.invoke('Can You Suummarize the video')

'**Video Summary**\n\n**Presenter:** Nitesh  \n**Channel:** Nitesh\'s YouTube Channel  \n**Series:** Agentic AI using LangGraph (Continuation of a chatbot development playlist)\n\n**Objective:**  \nAdd a **streaming feature** to the existing LangGraph-based chatbot to improve user experience by displaying responses token-by-token in real-time, rather than waiting for the full response to generate.\n\n---\n\n### Key Topics Covered\n\n1. **Problem Demonstration**  \n   - The current chatbot (with UI, short-term memory, and basic LLM interaction) works but **waits for the entire response** before displaying it.  \n   - Example: Asking the bot to "write a 500-word essay" causes a noticeable delay before any output appears.\n\n2. **Theoretical Foundation: What is Streaming?**  \n   - **Definition:** In LLMs, streaming means the model starts sending tokens as soon as they are generated, instead of waiting for the entire response to be ready.  \n   - **Why it’s needed:** Drastically improves 

In [196]:
qwen_model = ChatOllama(
    model="qwen3:8b"
)

## Full Structured Notes

In [71]:
from pydantic import BaseModel
from typing import List

class Topics(BaseModel):
    topics: List[str]

In [82]:

topic_prompt = PromptTemplate(
    template="""
You are an expert educational content analyst and curriculum designer.

Your task is to analyze the lecture transcript and extract a HIGHLY GRANULAR list of every single teachable topic. 

=========================
IMPORTANT LANGUAGE INSTRUCTIONS (STRICT):
=========================
- The transcript may be written in ANY language but you need to Give the all kind of Respone Only in English Language.
- You must internally translate everything to English.
- ALWAYS return the final topics in 100% professional, standard English.
- NEVER output any transliterated words or original language vocabulary.
- Do not mix languages. The output must be purely English.

=========================
TOPIC EXTRACTION RULES (STRICT):
=========================
1. BE EXHAUSTIVE: This is a long lecture. Extract EVERY specific concept, method, technique, framework, and theory. 
2. DO NOT GROUP: Do not merge distinct concepts. Every specific mechanism, step, or sub-topic must be extracted as its own separate topic But if a topic is short or Easily merge to the other topics then merge it with other most related topic.
3. VOLUME EXPECTATION: For long transcripts, you MUST extract approx 10 main most relevant topics to the video transceipt if need then increase the topic but not more try to merge most related topic and make it one most relevant topic.
4. NAMING CONVENTION: Use precise, dense technical names. Topic names MUST be between 2 and 7 words long. Do not use full sentences.
5. TEACHABILITY TEST: Only extract a topic if there is enough context in the transcript to write detailed notes about it. Do not extract passing mentions.
6. AVOID REDUNDANCY: If a topic is discussed multiple times, extract it only once when it is primarily taught. Do not list the same concept twice.
7. ORDER: Return topics in the exact chronological order they first appear in the transcript.
8. EXCLUSIONS: Ignore greetings, Q&A that goes off-topic, sponsor messages, general examples, and motivational talk.

BAD EXAMPLES (Too Broad or Too Long):
- AI
- understanding how loops work in python programming
- chatgpt examples

GOOD EXAMPLES (Specific & Granular):
- Retrieval-Augmented Generation (RAG) Architecture
- LangChain Expression Language (LCEL)
- Multi-head Attention Mechanism
- Similarity Search in ChromaDB

Transcript:
{transcript}
""",
    input_variables=["transcript"]
)


In [83]:

structured_model = model.with_structured_output(Topics)
topic_chain = topic_prompt | structured_model
result = topic_chain.invoke({
    "transcript": original_transcript
})

topic_list = result.topics

In [84]:
topic_prompt

PromptTemplate(input_variables=['transcript'], input_types={}, partial_variables={}, template='\nYou are an expert educational content analyst and curriculum designer.\n\nYour task is to analyze the lecture transcript and extract a HIGHLY GRANULAR list of every single teachable topic. \n\n=========================\nIMPORTANT LANGUAGE INSTRUCTIONS (STRICT):\n=========================\n- The transcript may be written in ANY language but you need to Give the all kind of Respone Only in English Language.\n- You must internally translate everything to English.\n- ALWAYS return the final topics in 100% professional, standard English.\n- NEVER output any transliterated words or original language vocabulary.\n- Do not mix languages. The output must be purely English.\n\n=========================\nTOPIC EXTRACTION RULES (STRICT):\n=========================\n1. BE EXHAUSTIVE: This is a long lecture. Extract EVERY specific concept, method, technique, framework, and theory. \n2. DO NOT GROUP: Do n

In [85]:
print(result.topics)

['Project Context and Feature Evolution', 'Non-Streaming UX Limitations Demo', 'Streaming Definition and Typewriter Effect', 'Streaming Benefits for LLM Applications', 'LangGraph Invoke vs Stream Mechanism', 'Python Generators for Token Yielding', 'Backend Streaming with Messages Mode', 'Streamlit Write Stream Integration', 'Session State Persistence for Streams', 'User Input Handling Bug Fix']


In [81]:
topic_list

['Non-Streaming Chatbot Latency Problems',
 'Token-by-Token Streaming Concept',
 'Streaming Typewriter Effect Demonstration',
 'LLM Streaming Technical Definition',
 'Streaming Benefit: Perceived Latency Reduction',
 'Streaming Benefit: Human Conversation Mimicry',
 'Streaming Benefit: Multimodal Voice UX Improvement',
 'Streaming Benefit: Long Code Output Readability',
 'Streaming Benefit: Mid-Generation Stop Capability',
 'Streaming Benefit: Agentic Process Visibility',
 'LangGraph Invoke vs Stream Methods',
 'Python Generator Yield Mechanics',
 'LangGraph Stream Method Arguments',
 'LangGraph Stream Mode Messages Configuration',
 'Stream Output Tuple Structure',
 'Generator Iteration for Token Printing',
 'Streamlit Chat UI Elements Reference',
 'Streamlit Write Stream Function Usage',
 'Graph Stream Integration with Write Stream',
 'Generator Expression Message Content Extraction',
 'Streamed Response Session State Storage',
 'User Input Variable Debugging Fix']

In [86]:
notes_prompt = """
You are an expert teacher, note-taking assistant, and educational content creator.
Your task is NOT to summarize.
Your task is to teach the student as if you are explaining the lecture in a classroom on the basis of the Given Topic and Context, which is a Youtube Video Transcript. Do not go outside the context; just use it and use a little of your creativity to explain this in a simple way with examples.
Create detailed study notes in professional Markdown format.

Topic:
{topic}

Transcript Context:
{context}

=========================
LANGUAGE REQUIREMENTS
=========================

- The provided context may be written in ANY language.
- First understand the meaning of the content in its original language.
- Translate and interpret the content internally using your language understanding capabilities.
- ALWAYS generate the final notes in clear, professional English.
- Never mix languages in the output.
- Never reproduce the notes in the original language.
- Preserve the original meaning, technical concepts, examples, and explanations while translating.
- If technical terms have no good translation, use the standard English technical term.
- The final notes should read as if the lecture was originally taught in English.

=========================
MARKDOWN REQUIREMENTS
=========================

- Return ONLY valid Markdown.
- Start with exactly ONE H1 heading which MUST be exactly: # {topic}
- Use H2 and H3 headings where appropriate.
- Use bullet points for important concepts.
- Use numbered lists for processes and steps.
- Use Markdown tables only when comparing concepts.
- Use **bold** for important terms and definitions.
- Use blockquotes (>) for key insights and reminders.
- Use *** as section separators.
- NEVER use YAML front matter and YAML metadata. This will create an error in the markdown file.
- NEVER output standalone --- lines.
- NEVER output metadata blocks.
- Keep formatting clean and readable.
- Do NOT wrap the output inside ```markdown blocks.
- Do NOT generate explanations outside Markdown.

=========================
EDUCATIONAL REQUIREMENTS
=========================

For the requested topic, include exactly this structure:

# {topic}

## What is it?

- Simple explanation first.
- Then technical explanation.

## Why do we need it?

- Explain the problem it solves.
- Explain why it is important.

## How does it work?

- Step-by-step explanation.
- Use numbered lists.

## Real World Example

- Give relatable examples from daily life.
- Use analogies whenever possible.

## Important Points

- Key concepts students should remember.

## Common Mistakes

- Mistakes beginners usually make.

## Interview Questions

- Generate 3-5 interview questions.

## Revision Notes

- Short revision bullets.
- Suitable for exam preparation.

=========================
QUALITY REQUIREMENTS
=========================

- MANDATORY: Your H1 heading must be the exact text provided in the "Topic:" section above. Do not invent a new overarching title based on the whole transcript.
- Focus ONLY on the specific `{topic}` provided. Do not write notes for the entire transcript.
- Expand concepts when necessary.
- Explain hidden assumptions.
- Preserve all important information related to the specific topic.
- Make the notes self-contained.
- A student should be able to learn the topic using only these notes.
- Prefer understanding over brevity.

Return only the final Markdown document.
"""

In [87]:
for i in range(0,len(topic_list)):
    print(f"{topic_list[i]} ########")

Project Context and Feature Evolution ########
Non-Streaming UX Limitations Demo ########
Streaming Definition and Typewriter Effect ########
Streaming Benefits for LLM Applications ########
LangGraph Invoke vs Stream Mechanism ########
Python Generators for Token Yielding ########
Backend Streaming with Messages Mode ########
Streamlit Write Stream Integration ########
Session State Persistence for Streams ########
User Input Handling Bug Fix ########


In [88]:
print(type(topic_list))

print(len(topic_list))

<class 'list'>
10


In [ ]:
import os

counter = 1

# 🛠️ Check for the file INSIDE the Markdown_files directory
while os.path.exists(f"outputs/markdown/lecture_notes{counter}.md"):
    counter += 1

# 📝 Create the unique filename
filename = f"outputs/markdown/lecture_notes{counter}.md"

with open(filename, "w", encoding="utf-8") as f:
    f.write("# Lecture Notes\n\n")

print(f"✅ Created: {filename}")

✅ Created: outputs/markdown/lecture_notes11.md


In [92]:
text_notes = ""

In [93]:
notes_Prompt_template =PromptTemplate(
    template= notes_prompt,
    input_variables=['topic','context']
)

notes_chain = notes_Prompt_template | model
for i in range(0,len(topic_list)):
    notes = notes_chain.invoke({
        "topic": topic_list[i],
        "context": original_transcript
    })
    text_notes += notes.content
    text_notes += "\n\n"   
    with open(filename, "a", encoding="utf-8") as f:
        f.write("\n---\n\n")
        f.write(notes.content.strip())
        f.write("\n\n")
    print(f"*                   @@@ START @@@ {topic_list[i]} @@@@ END @@@           *")
    print(notes.content)



*                   @@@ START @@@ Project Context and Feature Evolution @@@@ END @@@           *
# Project Context and Feature Evolution

## What is it?

- **Project Context**: This lecture continues a series on building an **Agentic AI Chatbot using LangGraph**. The project has evolved incrementally:
  1. **Basic Chatbot** — Direct LLM interaction with no memory.
  2. **Short-term Memory** — Chatbot remembers conversation history within a session.
  3. **UI Integration** — Added a Streamlit frontend for user interaction.
  4. **Current Feature: Streaming** — Solving the "all-at-once" response problem for long outputs.

- **Technical Definition of Streaming**:  
  In LLMs, **streaming** means the model sends tokens **as soon as they are generated**, instead of waiting for the entire response to be ready before returning it.

> **Core Difference**:  
> - **Non-streaming (Invoke)**: LLM thinks → generates full answer → sends complete response.  
> - **Streaming (Stream)**: LLM generates 

In [220]:
print(text_notes)        #This print the last updated notes.content

# Graph Processing Systems Overview

## What is it?

**Simple Explanation:**
A graph processing system allows you to model complex relationships and workflows (like a series of connected tasks) as a **graph**. Instead of running tasks sequentially or in isolated blocks, you define how different parts interact with each other through these connections.

**Technical Explanation:**
Graph processing systems are designed to handle computations on data structured as graphs, which consist of **nodes** (entities/data points) and **edges** (relationships between nodes). The system executes a defined workflow across this graph structure efficiently, often leveraging massive parallelization capabilities inspired by large-scale systems like Google Pregel.

***

## Why do we need it?

*   **Modeling Complexity:** Many real-world problems—such as social networks, routing in transportation systems, or dependency management in software builds—are inherently relational. Graph processing provides a natu

# Generate the .md file  --- TESTING

In [ ]:
# md_file_prompt = """

# You are an expert note-taking assistant.

# Convert the following text notes into a professional Markdown document. \n {text_notes}


# Requirements:

# - Output ONLY valid Markdown.
# - Start with a single H1 heading containing the topic.
# - Use H2 and H3 headings when needed.
# - Use bullet points for important concepts.
# - Use numbered lists for processes or steps.
# - Use tables only when comparing concepts.
# - Use **bold** for definitions and important terms.
# - Keep formatting clean and readable.
# - Remove filler words and repetition.
# - Expand explanations when necessary for learning.
# - Include examples when available.
# - Add a short summary section at the end.
# - Do NOT generate YAML metadata.
# - Do NOT generate explanations outside the Markdown.
# - Do NOT wrap the answer inside ```markdown blocks.

# Structure the notes naturally based on the content rather than forcing fixed sections.

# Return only the final Markdown document.
# """

In [ ]:
# md_file_prompt_template = PromptTemplate(
#     template= md_file_prompt,
#     input_variables= ['text_notes']
# )

In [ ]:
# md_file_prompt_template

PromptTemplate(input_variables=['text_notes'], input_types={}, partial_variables={}, template='\n\nYou are an expert note-taking assistant.\n\nConvert the following text notes into a professional Markdown document. \n {text_notes}\n\n\nRequirements:\n\n- Output ONLY valid Markdown.\n- Start with a single H1 heading containing the topic.\n- Use H2 and H3 headings when needed.\n- Use bullet points for important concepts.\n- Use numbered lists for processes or steps.\n- Use tables only when comparing concepts.\n- Use **bold** for definitions and important terms.\n- Keep formatting clean and readable.\n- Remove filler words and repetition.\n- Expand explanations when necessary for learning.\n- Include examples when available.\n- Add a short summary section at the end.\n- Do NOT generate YAML metadata.\n- Do NOT generate explanations outside the Markdown.\n- Do NOT wrap the answer inside ```markdown blocks.\n\nStructure the notes naturally based on the content rather than forcing fixed sect

In [ ]:
# md_file_chain = md_file_prompt_template | model | parser

# md_file_notes = md_file_chain.invoke({'text_notes':text_notes})

# print(md_file_notes)


# Bacteria vs. Viruses: A Comparative Guide

## Understanding Pathogens
This topic clarifies the fundamental differences between **bacteria** and **viruses**, two common causes of illness that are often confused. Knowing this distinction is critical because treating an infection with the wrong medicine (e.g., using antibiotics for a viral cold) is ineffective and can worsen the problem.

### Bacteria: The Living Cells
*   **Nature:** Bacteria are **prokaryotes**. This means they are single, complete cells that possess all the necessary machinery to reproduce independently.
*   **Reproduction:** They multiply by dividing their own cell walls (binary fission).
*   **Treatment:** Infections caused by bacteria can be targeted using **antibiotics**, which either kill the bacterial cells or inhibit their growth processes.

### Viruses: The Genetic Packets
*   **Nature:** Viruses are **not made of cells**. They are essentially inert packets consisting of genetic material (DNA or RNA) encased 

In [ ]:
# print(len(text_notes))
# print(text_notes)

41294
***Note: Given the massive amount of information in the source text, I have broken it down into four major topics to maintain clarity and adhere to your specified format. The language used is kept simple for easy understanding.***

---

# 🧬 Cell Cycle, Cancer, and Mutations #
## What is it?#
The cell cycle is the life process of how a normal cell grows, copies all its genetic material (DNA), and then divides into two new cells. A mutation is simply a change in the DNA code.
## Why do we need it?#
We need this process for growth (getting bigger), healing (replacing damaged skin or organs), and reproduction. The checkpoints are safety mechanisms that ensure every cell division is perfect.
## How does it work?#
1. **Interphase:** This is the "chilling" time. The cell grows and meticulously copies all its DNA.
2. **Checkpoints:** Proteins (like p53) act as quality control managers, checking if the cell is healthy and ready to divide. If something is wrong, the cell either fixes itsel

In [ ]:
# import os

# counter = 1
# while True:
#     filename_md = f"Markdown_files/lecture_notes{counter}.md"
#     if not os.path.exists(filename_md):
#         break
#     counter += 1
# with open(filename_md, "w", encoding="utf-8") as f:
#     f.write("# Lecture Notes\n\n")
    
# print(f"Created: {filename_md}")

Created: Markdown_files/md_file_lecture_notes3.md


In [91]:
# with open(filename, "a", encoding="utf-8") as f:
#         f.write(notes.content)

## Convert .md file to the PDF notes file

In [72]:
# import os

# counter = 1
# while True:
#     filename_md_notesPDF = f"Notes_PDF_files/md_file_lecture_notes{counter}.pdf"
#     if not os.path.exists(filename_md_notesPDF):
#         break
#     counter += 1

# print(filename_md_notesPDF)



# import os
# import re

# folder = "Notes_PDF_files"
# counter = 1

# for file in os.listdir(folder):
#     match = re.match(r"lecture_notes(\d+)\.pdf$", file)

#     if match:
#         counter = max(counter, int(match.group(1)) + 1)

# filename_notesPDF = f"{folder}/lecture_notes{counter}.pdf"


import os

folder = "Notes_PDF_files"
counter = 1

# 🛠️ Keep checking until we find a number that doesn't exist yet
while os.path.exists(f"{folder}/lecture_notes{counter}.pdf"):
    counter += 1

# 📝 Create the final unique filename
filename_notesPDF = f"{folder}/lecture_notes{counter}.pdf"

print(f"✅ Will create: {filename_notesPDF}")



✅ Will create: Notes_PDF_files/lecture_notes2.pdf


# Test

In [ ]:
import os

print(os.getcwd())
print(os.path.exists("Notes_PDF_files/md_file_lecture_notes2.pdf"))

/Users/asifkhan/Desktop/AI_ML/GenAI/LangChain/RAG_Project/YoTube_Intelligence
False


In [98]:


import os

print(os.getcwd())
print(os.path.exists("Markdown_files/lecture_notes5.md"))

/Users/asifkhan/Desktop/AI_ML/GenAI/LangChain/RAG_Project/YoTube_Intelligence
True


In [94]:
import os

for file in os.listdir("Notes_PDF_files"):
    print(file)

.DS_Store
lecture_notes1.pdf
md_file_lecture_notes6.pdf
md_file_lecture_notes4.pdf
md_file_lecture_notes3.pdf


In [ ]:
# import os

# print(os.listdir())

['Markdown_files', 'Notes_PDF_files', '.DS_Store', 'yt_intelligence_chatbot.ipynb', 'md_file_lecture_notes1.pdf', 'first_detailed_notes_by_YT_intelligence.pdf']


In [ ]:
# print(os.listdir("Notes_PDF_files"))

['.DS_Store', 'lecture_notes1.pdf', 'md_file_lecture_notes6.pdf', 'md_file_lecture_notes4.pdf', 'md_file_lecture_notes3.pdf']


# PDF Converter

In [96]:

import os

# folder = "output/pdf"
counter = 1

# 🛠️ Keep checking until we find a number that doesn't exist yet
while os.path.exists(f"outputs/pdf/lecture_notes{counter}.pdf"):
    counter += 1

# 📝 Create the final unique filename
filename_notesPDF = f"outputs/pdf/lecture_notes{counter}.pdf"

print(f"✅ Will create: {filename_notesPDF}")

✅ Will create: outputs/pdf/lecture_notes6.pdf


In [ ]:
# kaggle_filename = "Markdown_files/KAGGLE_WRITEUP.md"
# kaggle_filename1 = "Notes_PDF_files/KAGGLE_WRITEUP.pdf"

In [97]:
import subprocess

subprocess.run([
    "pandoc",
    filename,
    "--pdf-engine=xelatex",
    "-o",
    filename_notesPDF
])

[WARNING] Missing character: There is no ≠ (U+2260) (U+2260) in font [lmroman10-bold]:mapping=tex-
[WARNING] Missing character: There is no ≠ (U+2260) (U+2260) in font [lmroman10-bold]:mapping=tex-
[WARNING] Missing character: There is no ≈ (U+2248) (U+2248) in font [lmroman10-regular]:mapping=t


CompletedProcess(args=['pandoc', 'outputs/markdown/lecture_notes11.md', '--pdf-engine=xelatex', '-o', 'outputs/pdf/lecture_notes6.pdf'], returncode=0)

In [ ]:
# import subprocess

# try:
#     # 🚀 Run the Pandoc command
#     subprocess.run([
#         "pandoc",
#         filename,
#         "--pdf-engine=xelatex",
#         "--from=markdown-yaml_metadata_block",  # 🛠️ Tells Pandoc to stop looking for YAML metadata
#         "-o",
#         filename_notesPDF
#     ], check=True) # 🛠️ 'check=True' forces Python to catch the error if Pandoc fails
    
#     print(f"✅ Successfully created: {filename_notesPDF}")

# except subprocess.CalledProcessError as e:
#     # ⚠️ If Pandoc still fails, this block will handle the error
#     print(f"❌ Pandoc failed to convert the file. Error code: {e.returncode}")

✅ Successfully created: Notes_PDF_files/lecture_notes2.pdf


In [76]:
from markdown_pdf import MarkdownPdf, Section

# 📖 Read the contents of the file first
with open(filename, "r", encoding="utf-8") as file:
    md_text = file.read()

pdf = MarkdownPdf()

# 🛠️ Wrap the text inside a Section() object
pdf.add_section(Section(md_text))

# 💾 Save the file
pdf.save(f"{filename_notesPDF}_copy.pdf")

# PDF converter ---- TESTING

In [3]:
pip install markdown-pdf

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [markdown-pdf]

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
from markdown_pdf import MarkdownPdf

pdf = MarkdownPdf()
pdf.add_section(open(filename, encoding="utf-8").read())

pdf.save(filename_notesPDF)

NameError: name 'filename' is not defined

In [106]:
pip install markdown weasyprint


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
import markdown
from weasyprint import HTML

with open(filename, "r", encoding="utf-8") as f:
    md_text = f.read()

html = markdown.markdown(md_text)

HTML(string=html).write_pdf(filename_notesPDF)


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----



OSError: cannot load library 'libgobject-2.0-0': dlopen(libgobject-2.0-0, 0x0002): tried: 'libgobject-2.0-0' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibgobject-2.0-0' (no such file), '/usr/lib/libgobject-2.0-0' (no such file, not in dyld cache), 'libgobject-2.0-0' (no such file).  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0'

In [1]:
from weasyprint import HTML

HTML(string="<h1>Hello</h1>").write_pdf("test.pdf")


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----



OSError: cannot load library 'libgobject-2.0-0': dlopen(libgobject-2.0-0, 0x0002): tried: 'libgobject-2.0-0' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibgobject-2.0-0' (no such file), '/usr/lib/libgobject-2.0-0' (no such file, not in dyld cache), 'libgobject-2.0-0' (no such file).  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0'

In [2]:
import ctypes.util

print(ctypes.util.find_library("gobject-2.0"))
print(ctypes.util.find_library("pango-1.0"))
print(ctypes.util.find_library("cairo"))

/usr/local/lib/libgobject-2.0.dylib
/usr/local/lib/libpango-1.0.dylib
/usr/local/lib/libcairo.dylib


In [ ]:
# import subprocess

# result = subprocess.run(
#     ["pandoc", "lecture_notes.md", "-o", "lecture_notes.pdf"],
#     capture_output=True,
#     text=True
# )

# print(result.stderr)

Error parsing YAML metadata at "lecture_notes.md" (line 332, column 1):
YAML parse exception at line 2, column 23,
while scanning an alias:
did not find expected alphabetic or numeric character



In [62]:
pip install markdown weasyprint

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 4.0 MB/s eta 0:00:00a 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7/7 [weasyprint]7 [weasyprint]

[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [ ]:
# from markdown import markdown
# from weasyprint import HTML

# # Read markdown file
# with open(filename, "r", encoding="utf-8") as f:
#     md_content = f.read()

# # Convert Markdown to HTML
# html_body = markdown(
#     md_content,
#     extensions=["tables", "fenced_code", "toc"]
# )

# # Beautiful but professional styling
# html_template = f"""
# <!DOCTYPE html>
# <html>
# <head>
# <meta charset="utf-8">

# <style>

# body {{
#     font-family: "Segoe UI", Arial, sans-serif;
#     line-height: 1.8;
#     max-width: 900px;
#     margin: 40px auto;
#     padding: 20px;
#     color: #2c3e50;
# }}

# h1 {{
#     color: #1e3a5f;
#     border-bottom: 3px solid #4f81bd;
#     padding-bottom: 8px;
#     margin-top: 30px;
# }}

# h2 {{
#     color: #2c5282;
#     margin-top: 25px;
# }}

# h3 {{
#     color: #3b82f6;
# }}

# strong {{
#     color: #1f2937;
# }}

# p {{
#     text-align: justify;
# }}

# ul, ol {{
#     margin-left: 20px;
# }}

# li {{
#     margin-bottom: 8px;
# }}

# blockquote {{
#     border-left: 4px solid #4f81bd;
#     background: #f8fafc;
#     padding: 10px 15px;
#     margin: 15px 0;
# }}

# code {{
#     background: #f3f4f6;
#     padding: 2px 5px;
#     border-radius: 4px;
#     font-family: Consolas, monospace;
# }}

# pre {{
#     background: #f8f9fa;
#     border: 1px solid #ddd;
#     padding: 12px;
#     border-radius: 8px;
#     overflow-x: auto;
# }}

# table {{
#     border-collapse: collapse;
#     width: 100%;
#     margin: 15px 0;
# }}

# th {{
#     background: #4f81bd;
#     color: white;
# }}

# th, td {{
#     border: 1px solid #ddd;
#     padding: 10px;
# }}

# tr:nth-child(even) {{
#     background: #f8fafc;
# }}

# @page {{
#     size: A4;
#     margin: 1in;
# }}

# </style>

# </head>
# <body>

# {html_body}

# </body>
# </html>
# """

# # Generate PDF
# HTML(string=html_template).write_pdf("lecture_notes1.pdf")

# print("PDF created successfully!")


-----

WeasyPrint could not import some external libraries. Please carefully follow the installation steps before reporting an issue:
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#installation
https://doc.courtbouillon.org/weasyprint/stable/first_steps.html#troubleshooting 

-----



OSError: cannot load library 'libgobject-2.0-0': dlopen(libgobject-2.0-0, 0x0002): tried: 'libgobject-2.0-0' (no such file), '/System/Volumes/Preboot/Cryptexes/OSlibgobject-2.0-0' (no such file), '/usr/lib/libgobject-2.0-0' (no such file, not in dyld cache), 'libgobject-2.0-0' (no such file).  Additionally, ctypes.util.find_library() did not manage to locate a library called 'libgobject-2.0-0'

In [64]:
pip install weasyprint


[notice] A new release of pip is available: 25.1.1 -> 26.1.2
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


# Timestamp recommmend

In [98]:
transcript_list

FetchedTranscript(snippets=[FetchedTranscriptSnippet(text='हाय गाइज़, माय नेम इज़ नितेश एंड यू आर', start=0.0, duration=4.16), FetchedTranscriptSnippet(text='वेलकम टू माय YouTube चैनल। इस वीडियो में', start=1.92, duration=4.56), FetchedTranscriptSnippet(text='भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग', start=4.16, duration=4.56), FetchedTranscriptSnippet(text='ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो', start=6.48, duration=4.72), FetchedTranscriptSnippet(text='पिछले कुछ वीडियोस में जैसा हमने प्लान', start=8.72, duration=5.6), FetchedTranscriptSnippet(text='किया था हम एक चैटबॉट डेवलप कर रहे हैं।', start=11.2, duration=6.24), FetchedTranscriptSnippet(text='और धीरे-धीरे उस चैटबॉट में हम फीचर्स और', start=14.32, duration=5.6), FetchedTranscriptSnippet(text='ऐड करते जा रहे हैं। सबसे पहले हमने एक', start=17.44, duration=4.24), FetchedTranscriptSnippet(text='बेसिक चैटबॉट बनाया था। जहां पे आप एक', start=19.92, duration=4.64), FetchedTranscriptSnippet(text='एलएलएम से बात कर पा रहे थे। फिर उसी में', start=

In [99]:
transcript_with_timestamp = []

In [100]:
for i in range(0,len(transcript_list)):
    temp = []
    print(i)
    temp.append(f"text : {transcript_list[i].text}")
    temp.append(f"Start : {(transcript_list[i].start)/60:.2f} minute")
    temp.append(f"End : {((transcript_list[i].start)+transcript_list[i].duration)/60:.2f} minute")
    transcript_with_timestamp.append(temp)


0
1
2
3
4
5
6
7
8
9
10
11
12
13
14
15
16
17
18
19
20
21
22
23
24
25
26
27
28
29
30
31
32
33
34
35
36
37
38
39
40
41
42
43
44
45
46
47
48
49
50
51
52
53
54
55
56
57
58
59
60
61
62
63
64
65
66
67
68
69
70
71
72
73
74
75
76
77
78
79
80
81
82
83
84
85
86
87
88
89
90
91
92
93
94
95
96
97
98
99
100
101
102
103
104
105
106
107
108
109
110
111
112
113
114
115
116
117
118
119
120
121
122
123
124
125
126
127
128
129
130
131
132
133
134
135
136
137
138
139
140
141
142
143
144
145
146
147
148
149
150
151
152
153
154
155
156
157
158
159
160
161
162
163
164
165
166
167
168
169
170
171
172
173
174
175
176
177
178
179
180
181
182
183
184
185
186
187
188
189
190
191
192
193
194
195
196
197
198
199
200
201
202
203
204
205
206
207
208
209
210
211
212
213
214
215
216
217
218
219
220
221
222
223
224
225
226
227
228
229
230
231
232
233
234
235
236
237
238
239
240
241
242
243
244
245
246
247
248
249
250
251
252
253
254
255
256
257
258
259
260
261
262
263
264
265
266
267
268
269
270
271
272
273
274
275
276
27

In [101]:
transcript_with_timestamp

[['text : हाय गाइज़, माय नेम इज़ नितेश एंड यू आर',
  'Start : 0.00 minute',
  'End : 0.07 minute'],
 ['text : वेलकम टू माय YouTube चैनल। इस वीडियो में',
  'Start : 0.03 minute',
  'End : 0.11 minute'],
 ['text : भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग',
  'Start : 0.07 minute',
  'End : 0.15 minute'],
 ['text : ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो',
  'Start : 0.11 minute',
  'End : 0.19 minute'],
 ['text : पिछले कुछ वीडियोस में जैसा हमने प्लान',
  'Start : 0.15 minute',
  'End : 0.24 minute'],
 ['text : किया था हम एक चैटबॉट डेवलप कर रहे हैं।',
  'Start : 0.19 minute',
  'End : 0.29 minute'],
 ['text : और धीरे-धीरे उस चैटबॉट में हम फीचर्स और',
  'Start : 0.24 minute',
  'End : 0.33 minute'],
 ['text : ऐड करते जा रहे हैं। सबसे पहले हमने एक',
  'Start : 0.29 minute',
  'End : 0.36 minute'],
 ['text : बेसिक चैटबॉट बनाया था। जहां पे आप एक',
  'Start : 0.33 minute',
  'End : 0.41 minute'],
 ['text : एलएलएम से बात कर पा रहे थे। फिर उसी में',
  'Start : 0.36 minute',
  'End : 0.45 minute'],
 ['tex

# Convert to Document

In [102]:
from langchain_core.documents import Document

timestamp_docs = []

for i in range(len(transcript_list)):
    timestamp_docs.append(
        Document(
            page_content=transcript_list[i].text,
            metadata={
                "start": round(transcript_list[i].start / 60, 2),
                "end": round(
                    (transcript_list[i].start + transcript_list[i].duration) / 60,
                    2
                )
            }
        )
    )

In [103]:
timestamp_docs

[Document(metadata={'start': 0.0, 'end': 0.07}, page_content='हाय गाइज़, माय नेम इज़ नितेश एंड यू आर'),
 Document(metadata={'start': 0.03, 'end': 0.11}, page_content='वेलकम टू माय YouTube चैनल। इस वीडियो में'),
 Document(metadata={'start': 0.07, 'end': 0.15}, page_content='भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग'),
 Document(metadata={'start': 0.11, 'end': 0.19}, page_content='ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो'),
 Document(metadata={'start': 0.15, 'end': 0.24}, page_content='पिछले कुछ वीडियोस में जैसा हमने प्लान'),
 Document(metadata={'start': 0.19, 'end': 0.29}, page_content='किया था हम एक चैटबॉट डेवलप कर रहे हैं।'),
 Document(metadata={'start': 0.24, 'end': 0.33}, page_content='और धीरे-धीरे उस चैटबॉट में हम फीचर्स और'),
 Document(metadata={'start': 0.29, 'end': 0.36}, page_content='ऐड करते जा रहे हैं। सबसे पहले हमने एक'),
 Document(metadata={'start': 0.33, 'end': 0.41}, page_content='बेसिक चैटबॉट बनाया था। जहां पे आप एक'),
 Document(metadata={'start': 0.36, 'end': 0.45}, page_content='

In [104]:
large_trans_window = []


In [ ]:
# Chunking by itself on the timestamps

In [105]:

for i in range(0,len(transcript_list),10):
    print(f"i is {i}")
    page_text_content = ""
    k = i+10
    if(k>=len(transcript_list)):
        k=len(transcript_list) - 1
    for j in range(i,k):
        print(f"j is {j}")
        page_text_content = f"{page_text_content + transcript_list[j].text} "

    large_trans_window.append(
        Document(
            page_content=page_text_content,
            metadata={
                "start": round(transcript_list[i].start / 60, 2),
                "end": round(
                    (transcript_list[k].start + transcript_list[k].duration) / 60,2)
            }
        )
    )

i is 0
j is 0
j is 1
j is 2
j is 3
j is 4
j is 5
j is 6
j is 7
j is 8
j is 9
i is 10
j is 10
j is 11
j is 12
j is 13
j is 14
j is 15
j is 16
j is 17
j is 18
j is 19
i is 20
j is 20
j is 21
j is 22
j is 23
j is 24
j is 25
j is 26
j is 27
j is 28
j is 29
i is 30
j is 30
j is 31
j is 32
j is 33
j is 34
j is 35
j is 36
j is 37
j is 38
j is 39
i is 40
j is 40
j is 41
j is 42
j is 43
j is 44
j is 45
j is 46
j is 47
j is 48
j is 49
i is 50
j is 50
j is 51
j is 52
j is 53
j is 54
j is 55
j is 56
j is 57
j is 58
j is 59
i is 60
j is 60
j is 61
j is 62
j is 63
j is 64
j is 65
j is 66
j is 67
j is 68
j is 69
i is 70
j is 70
j is 71
j is 72
j is 73
j is 74
j is 75
j is 76
j is 77
j is 78
j is 79
i is 80
j is 80
j is 81
j is 82
j is 83
j is 84
j is 85
j is 86
j is 87
j is 88
j is 89
i is 90
j is 90
j is 91
j is 92
j is 93
j is 94
j is 95
j is 96
j is 97
j is 98
j is 99
i is 100
j is 100
j is 101
j is 102
j is 103
j is 104
j is 105
j is 106
j is 107
j is 108
j is 109
i is 110
j is 110
j is 111
j is 

In [106]:
large_trans_window

[Document(metadata={'start': 0.0, 'end': 0.5}, page_content='हाय गाइज़, माय नेम इज़ नितेश एंड यू आर वेलकम टू माय YouTube चैनल। इस वीडियो में भी हम लोग अपना एजेंटिक एआई यूज़िंग लंग ग्राफ प्लेलिस्ट कंटिन्यू करेंगे। सो पिछले कुछ वीडियोस में जैसा हमने प्लान किया था हम एक चैटबॉट डेवलप कर रहे हैं। और धीरे-धीरे उस चैटबॉट में हम फीचर्स और ऐड करते जा रहे हैं। सबसे पहले हमने एक बेसिक चैटबॉट बनाया था। जहां पे आप एक एलएलएम से बात कर पा रहे थे। फिर उसी में '),
 Document(metadata={'start': 0.41, 'end': 0.94}, page_content='हमने शॉर्ट टर्म मेमोरी का फीचर ऐड किया। सो दैट हमारा चैटबॉट हमारे पास इंटरेक्शंस याद रख पाए। उसके बाद हमने उस चैटबॉट को एक यूआई दिया। आज हम अपने चैटबॉट की एक और प्रॉब्लम सॉल्व करने जा रहे हैं। सबसे पहले मैं आपको वो प्रॉब्लम दिखाता हूं और फिर मैं आपको बताता हूं कि क्या सॉल्यूशन है उस प्रॉब्लम को सॉल्व करने का। ठीक है? तो स्क्रीन पे अभी आपको हमारे चैटबॉट का यूआई दिख रहा होगा। सो यहां पे एक बार '),
 Document(metadata={'start': 0.88, 'end': 1.47}, page_content='पहले चेक कर लेते हैं। चैटब

In [107]:
vector_store_1 = FAISS.from_documents(large_trans_window,embeddingmodel)

In [108]:
vector_store_1.index_to_docstore_id

{0: '99c04ca9-03d1-4c0b-a040-26859bfa76f0',
 1: '4522b5b1-8c6b-4f20-9db4-9b2d154893ae',
 2: '401c51d7-2ad6-4427-82e9-ffb134b14be2',
 3: 'fecceb55-f106-444a-8580-58f77d0a446d',
 4: '21d18b49-0413-49bc-b2dd-dbb8925ef4d5',
 5: 'cc674f90-80f7-4129-985d-6329bc660f3e',
 6: '0875c8aa-bd36-4dd2-a87c-b320300deb33',
 7: 'b80a63fc-ee4c-4b45-aa5d-13ba9e7657e6',
 8: 'a33c3f21-af3a-4cc1-a5b3-fbdd4a9dfcd9',
 9: '18e86c77-f618-431b-9537-cf88f077598e',
 10: '53cfe317-0417-4711-88f7-74cf89b5cc3c',
 11: '403125d8-99fd-4ed7-af38-513a83bc61d3',
 12: 'dd9dfc18-f07d-45d9-8733-e7cabdc64133',
 13: '91c39c24-cea5-4c83-a8e5-35f1fae32495',
 14: '7aa48424-7590-4f8d-b2c2-1f320a5eea15',
 15: '20f6de5d-2909-4897-8a4d-5cda820a40f4',
 16: 'd8d0f2d6-f69a-4372-8485-9389e74a1661',
 17: 'ef9b7650-aa49-428c-a503-c0a550aefad6',
 18: 'c557ffe8-d4a7-4e34-acdf-9060e6154b53',
 19: 'a7214b81-0713-4b5d-909c-aeaa9d6ed8e0',
 20: 'dd394a05-97c3-4868-ad69-cbfb68508ba3',
 21: 'ce74dbd5-c522-4b39-968d-990b36d9d84a',
 22: '25e448e4-1711-

In [109]:
vector_store_1.get_by_ids(['217fc2fb-e79a-4fe6-8518-d5936ac3ef5f'])

[]

In [110]:
retriever_1 = vector_store_1.as_retriever(search_type="similarity",search_kwargs={"k":4})

In [111]:
retriever_1

VectorStoreRetriever(tags=['FAISS', 'NVIDIAEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x12fea1590>, search_kwargs={'k': 4})

In [112]:
#Testing the retriever_1
retriever_1.invoke("What is the Streaming")

[Document(id='337f9d0c-a832-46ed-a6f7-e832adeacabe', metadata={'start': 19.28, 'end': 19.83}, page_content='आपको बताता हूं। आपको बेसिकली ये जो कोड है जहां पे आप ग्राफ डॉट स्ट्रीम कर रहे हो उसको लेके आना है इंस्टेड ऑफ़ इनवोक। ठीक है? और ये पूरी स्ट्रीमिंग को करने के लिए हमें एक यूआई एलिमेंट लगेगा। अगर आप स्ट्रीमलेट के डॉक्स पर जाओगे तो वहां पर आपको सारे के सारे चैट एलिमेंट्स दिखाई देंगे जो भी आपके पास अवेलेबल है स्ट्रीमलेट में। इनमें से दो हमने ऑलरेडी यूज़ कर रखे हैं। चैट इनपुट चैट मैसेज ठीक '),
 Document(id='f0517a85-2957-4415-8ee5-6b1a99c13630', metadata={'start': 24.45, 'end': 24.97}, page_content='मेक पास्ता या पिज़्ज़ा। ठीक है? सो नाउ दिस वन इज़ वर्किंग। ठीक है? तो, हमने हमारे चार्ट बॉट में एक छोटा सा बट बहुत इंपॉर्टेंट फीचर ऐड किया स्ट्रीमिंग का। इस प्रोसेस में हमने यह भी सीखा कि स्ट्रीमिंग होता क्या है? टेक्निकली स्ट्रीमिंग काम कैसे करता है? और स्ट्रीमिंग क्यों जरूरी है? ठीक है? तो, आई होप आपको यह वीडियो पसंद आया। और अगर वीडियो पसंद आया तो आप प्लीज लाइक करना। '),
 Document(id='dd9dfc1

In [113]:
prompt_1 = PromptTemplate(
    template="""You are a helpful assistant.
Answer the user's question using ONLY the provided transcript context. 

Follow these strict formatting rules:
1. First, answer the question clearly. Whenever you see in the context written like [Timestamp: <any number> - <any number> ] its mean this is not the part of of textual response this is the Timestamp of the video which is provided by the user So,  Do NOT include any timestamps in this part of your Text answer , The Timestaps store somewhere and use all at once and Put this timestamp at the end of your response with the Accending order of the numerical values group in timestamp on the basis of first numeric value of each timestamp to arrange its in assending order.
2. At the very end of your response, on a new line, add the references exactly like this: "If you need to know more, go to this timestamp: [insert start and end times here] if more than one then use this method again again".

If the context is insufficient, just say you don't know.

Context:
{context}

Question: {question}
Answer:""",
    input_variables=['context', 'question']
)

In [114]:
question_1          = "What is the streaming in LangGraph"
retrieved_docs_1    = retriever_1.invoke(question_1)

In [115]:
retrieved_docs_1

[Document(id='18e86c77-f618-431b-9537-cf88f077598e', metadata={'start': 4.0, 'end': 4.46}, page_content='रिक्वायर्ड होता है? फिर हम कोड में जाकर के चेंजेस करेंगे कि कैसे स्ट्रीमिंग को इंप्लीमेंट किया जा सकता है लंग ग्राफ में। ठीक है? तो आई होप आपको इस वीडियो का पूरा गोल समझ में आ गया। नाउ लेट्स स्टार्ट द वीडियो। तो गाइज़ एक बार फॉर्मली डिस्कस कर लेते हैं कि स्ट्रीमिंग होता क्या है? और स्ट्रीमिंग की जरूरत क्यों है हमें एलएलएम बेस्ड एप्लीकेशनेशंस बनाने के लिए। सो यहां पे मैंने एक डेफिनेशन लिखा '),
 Document(id='337f9d0c-a832-46ed-a6f7-e832adeacabe', metadata={'start': 19.28, 'end': 19.83}, page_content='आपको बताता हूं। आपको बेसिकली ये जो कोड है जहां पे आप ग्राफ डॉट स्ट्रीम कर रहे हो उसको लेके आना है इंस्टेड ऑफ़ इनवोक। ठीक है? और ये पूरी स्ट्रीमिंग को करने के लिए हमें एक यूआई एलिमेंट लगेगा। अगर आप स्ट्रीमलेट के डॉक्स पर जाओगे तो वहां पर आपको सारे के सारे चैट एलिमेंट्स दिखाई देंगे जो भी आपके पास अवेलेबल है स्ट्रीमलेट में। इनमें से दो हमने ऑलरेडी यूज़ कर रखे हैं। चैट इनपुट चैट मैसेज ठीक '),
 D

In [116]:
context_text_1 = ""

for doc in retrieved_docs_1:
    # Safely extract the start and end times from the metadata dictionary
    start_time = doc.metadata.get('start', 'N/A')
    end_time = doc.metadata.get('end', 'N/A')
    
    # Format with the timestamp on top, followed by the text and a blank line
    context_text_1 += f"[Timestamp: {start_time} - {end_time}]\n{doc.page_content}\n\n"

# You can print it to verify the clean output
print(context_text_1)

[Timestamp: 4.0 - 4.46]
रिक्वायर्ड होता है? फिर हम कोड में जाकर के चेंजेस करेंगे कि कैसे स्ट्रीमिंग को इंप्लीमेंट किया जा सकता है लंग ग्राफ में। ठीक है? तो आई होप आपको इस वीडियो का पूरा गोल समझ में आ गया। नाउ लेट्स स्टार्ट द वीडियो। तो गाइज़ एक बार फॉर्मली डिस्कस कर लेते हैं कि स्ट्रीमिंग होता क्या है? और स्ट्रीमिंग की जरूरत क्यों है हमें एलएलएम बेस्ड एप्लीकेशनेशंस बनाने के लिए। सो यहां पे मैंने एक डेफिनेशन लिखा 

[Timestamp: 19.28 - 19.83]
आपको बताता हूं। आपको बेसिकली ये जो कोड है जहां पे आप ग्राफ डॉट स्ट्रीम कर रहे हो उसको लेके आना है इंस्टेड ऑफ़ इनवोक। ठीक है? और ये पूरी स्ट्रीमिंग को करने के लिए हमें एक यूआई एलिमेंट लगेगा। अगर आप स्ट्रीमलेट के डॉक्स पर जाओगे तो वहां पर आपको सारे के सारे चैट एलिमेंट्स दिखाई देंगे जो भी आपके पास अवेलेबल है स्ट्रीमलेट में। इनमें से दो हमने ऑलरेडी यूज़ कर रखे हैं। चैट इनपुट चैट मैसेज ठीक 

[Timestamp: 12.25 - 12.79]
हूं। जहां पर इन लोगों ने बताया है कि कैसे आप स्ट्रीमिंग इंप्लीमेंट कर सकते हो। और इस गिवन एग्जांपल में आप आसानी से देख सकते हो कि जो इकलौता च

In [117]:
final_prompt_1 = prompt_1.invoke({"context": context_text_1, "question": question_1})

In [118]:
final_prompt_1

StringPromptValue(text='You are a helpful assistant.\nAnswer the user\'s question using ONLY the provided transcript context. \n\nFollow these strict formatting rules:\n1. First, answer the question clearly. Whenever you see in the context written like [Timestamp: <any number> - <any number> ] its mean this is not the part of of textual response this is the Timestamp of the video which is provided by the user So,  Do NOT include any timestamps in this part of your Text answer , The Timestaps store somewhere and use all at once and Put this timestamp at the end of your response with the Accending order of the numerical values group in timestamp on the basis of first numeric value of each timestamp to arrange its in assending order.\n2. At the very end of your response, on a new line, add the references exactly like this: "If you need to know more, go to this timestamp: [insert start and end times here] if more than one then use this method again again".\n\nIf the context is insufficient

In [119]:
answer_1 = model.invoke(final_prompt_1)


In [120]:
print(answer_1.content)

Based on the provided context, streaming in LangGraph is implemented by calling the `graph.stream()` function instead of `graph.invoke()` to execute the graph. This allows for streaming responses in LLM-based applications. It requires a configuration object that includes a `thread_id` and a `stream_mode` parameter (with multiple modes available, such as "updates"). The streaming functionality integrates with UI elements like Streamlit's chat components (chat_input, chat_message) to display the streamed output.

If you need to know more, go to this timestamp: [4.0 - 4.46]
If you need to know more, go to this timestamp: [12.25 - 12.79]
If you need to know more, go to this timestamp: [15.12 - 15.74]
If you need to know more, go to this timestamp: [19.28 - 19.83]
